Extraindo dados com SCRAPY SHELL
- Extrair dados do https://quotes.toscrape.com/
- Entender melhor os conceitos antes de aplica-los no spider

Sabendo que no site, cada citaçao tem
- Texto
-Autor
-Tags

Windows:

C:\> scrapy shell "https://quotes.toscrape.com/page/1/"

Ele abre o prompt do scrapy shell e no terminal pode acessar os elementos da pagina

Possiveis comandos para colocar no terminal:
- response.css('title')
    SAIDA: [<Selector query='descendant-or-self::title' data='<title>Quotes to Scrape</title>'>]


- response.css('title::text').extract
    SAIDA: [<Selector query='descendant-or-self::title/text()' data='Quotes to Scrape'>]

- response.css('link').extract()
    SAIDA: ['<link rel="stylesheet" href="/static/bootstrap.min.css">', '<link rel="stylesheet" href="/static/main.css">']


- response.css('link').extract_first()
    SAIDA: '<link rel="stylesheet" href="/static/bootstrap.min.css">'


# usando expressoes regulares (re)
- response.css('title::text').re(r'Quotes.*')
    SAIDA: ['Quotes to Scrape']

- response.css('span::text').re(r'The Person.*')

Extrair as citaçoes e os autores 
- Cada citação tem essa estrutura
aula7_imagem.png

<div class="quote">
    <span class="text"> ...... <\span>
    <span>
        by <small class="author"> ..... <\small>
        <a href="/author/....">> (about) <\a>
    </span>

    <div class="tags">
        Tags:
        <a class="tag" href="...."> ..... </a>
        .....

div.quote
span.text
small.author
div.tags
a.tag
span.text::text
small.author::text
a.tag::text

- citacao0 = response.css("div.quote")[0]
- texto = citacao0.css("span.text::text").extract_first()
- texto
    SAIDA: '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

- autor = citacao0.css("small.author::text").extract_first()
- autor
    SAIDA: 'Albert Einstein'

- tags = citacao0.css("div.tags a.tag::text").extract()
- tags
    SAIDA: ['change', 'deep-thoughts', 'thinking', 'world']

# iterar sobre todas as citacoes e colocar num dicionario
for citacao in response.css("div.quote"):
    texto = citacao.css("span.text::text").extract()
    autor = citacao.css("small.author::text").extract()
    tags = citacao.css("div.tags a.tag::text").extract()

    dicionario = dict(texto=texto, autor=autor, tags=tags)

    print(dicionario)

SAIDA::

{'texto': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'autor': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}
{'texto': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'autor': 'J.K. Rowling', 'tags': ['abilities', 'choices']}
{'texto': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'autor': 'Albert Einstein', 'tags': ['inspirational', 'life', 'live', 'miracle', 'miracles']}
{'texto': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', 'autor': 'Jane Austen', 'tags': ['aliteracy', 'books', 'classic', 'humor']}
{'texto': "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", 'autor': 'Marilyn Monroe', 'tags': ['be-yourself', 'inspirational']}
{'texto': '“Try not to become a man of success. Rather become a man of value.”', 'autor': 'Albert Einstein', 'tags': ['adulthood', 'success', 'value']}
{'texto': '“It is better to be hated for what you are than to be loved for what you are not.”', 'autor': 'André Gide', 'tags': ['life', 'love']}
{'texto': "“I have not failed. I've just found 10,000 ways that won't work.”", 'autor': 'Thomas A. Edison', 'tags': ['edison', 'failure', 'inspirational', 'paraphrased']}
{'texto': "“A woman is like a tea bag; you never know how strong it is until it's in hot water.”", 'autor': 'Eleanor Roosevelt', 'tags': ['misattributed-eleanor-roosevelt']}
{'texto': '“A day without sunshine is like, you know, night.”', 'autor': 'Steve Martin', 'tags': ['humor', 'obvious', 'simile']}
>>>

NO TERMINAL:
- scrapy startproject aulascrapy2
O PROPRIO TERMINAL CRIOU A PASTA

# CRIEI O 'citacoes3.py' em aulascrapy2/spiders

import scrapy

class CitacoesSpider(scrapy.Spider):
    name = "citacoes3"

    start_urls = [ "https://quotes.toscrape.com/page/1/", "https://quotes.toscrape.com/page/2/"]

    def parse(self, response):
        for citacao in response.css('div.quote'):
            yield {
                "texto": citacao.css('span.text::text').extract_first(),
                "autor": citacao.css('small.author::text').extract_first(),
                "tags": citacao.css('div.tags a.tag::text').extract()
            }

PS C:\Users\anacl\OneDrive\Área de Trabalho\cursos\web_scraping\4_Scrapy\aulascrapy\aulascrapy2> scrapy crawl citacoes3

SAIDA:

2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”', 'autor': 'Albert Einstein', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', 'autor': 'J.K. Rowling', 'tags': ['abilities', 'choices']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”', 'autor': 'Albert Einstein', 'tags': ['inspirational', 'life', 'live', 'miracle', 'miracles']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', 'autor': 'Jane Austen', 'tags': ['aliteracy', 'books', 'classic', 'humor']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", 'autor': 'Marilyn Monroe', 'tags': ['be-yourself', 'inspirational']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“Try not to become a man of success. Rather become a man of value.”', 'autor': 'Albert Einstein', 'tags': ['adulthood', 'success', 'value']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“It is better to be hated for what you are than to be loved for what you are not.”', 'autor': 'André Gide', 'tags': ['life', 'love']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': "“I have not failed. I've just found 10,000 ways that won't work.”", 'autor': 'Thomas A. Edison', 'tags': ['edison', 'failure', 'inspirational', 'paraphrased']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': "“A woman is like a tea bag; you never know how strong it is until it's in hot water.”", 'autor': 'Eleanor Roosevelt', 'tags': ['misattributed-eleanor-roosevelt']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/1/>
{'texto': '“A day without sunshine is like, you know, night.”', 'autor': 'Steve Martin', 'tags': ['humor', 'obvious', 'simile']}
2026-03-04 21:53:10 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://quotes.toscrape.com/page/2/> (referer: None)
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': "“This life is what you make it. No matter what, you're going to mess up sometimes, it's a universal truth. But the good part is you get to decide how you're going to mess it up. Girls will be your friends - they'll act like it anyway. But just remember, some come, some go. The ones that stay with you through everything - they're your true best friends. Don't let go of them. Also remember, sisters make the best friends in the world. As for lovers, well, they'll come and go too. And baby, I hate to say it, most of them - actually pretty much all of them are going to break your heart, but you can't give up because if you give up, you'll never find your soulmate. You'll never find that half who makes you whole and that goes for everything. Just because you fail once, doesn't mean you're gonna fail at everything. Keep trying, hold on, and always, always, always believe in yourself, because if you don't, then who will, sweetie? So keep your head high, keep your chin up, and most importantly, keep smiling, because life's a beautiful thing and there's so much to smile about.”", 'autor': 'Marilyn Monroe', 'tags': ['friends', 'heartbreak', 'inspirational', 'life', 'love', 'sisters']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': '“It takes a great deal of bravery to stand up to our enemies, but just as much to stand up to our friends.”', 'autor': 'J.K. Rowling', 'tags': ['courage', 'friends']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': "“If you can't explain it to a six year old, you don't understand it yourself.”", 'autor': 'Albert Einstein', 'tags': ['simplicity', 'understand']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': "“You may not be her first, her last, or her only. She loved before she may love again. But if she loves you now, what else matters? She's not perfect—you aren't either, and the two of you may never be perfect together but if she can make you laugh, cause you to think twice, and admit to being human and making mistakes, hold onto her and give her the most you can. She may not be thinking about you every second of the day, but she will give you a part of her that she knows you can break—her heart. So don't hurt her, don't change her, don't analyze and don't expect more than she can give. Smile when she makes you happy, let her know when she makes you mad, and miss her when she's not there.”", 'autor': 'Bob Marley', 'tags': ['love']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': '“I like nonsense, it wakes up the brain cells. Fantasy is a necessary ingredient in living.”', 'autor': 'Dr. Seuss', 'tags': ['fantasy']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': '“I may not have gone where I intended to go, but I think I have ended up where I needed to be.”', 'autor': 'Douglas Adams', 'tags': ['life', 'navigation']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': "“The opposite of love is not hate, it's indifference. The opposite of art is not ugliness, it's indifference. The opposite of faith is not heresy, it's indifference. And the opposite of life is not death, it's indifference.”", 'autor': 'Elie Wiesel', 'tags': ['activism', 'apathy', 'hate', 'indifference', 'inspirational', 'love', 'opposite', 'philosophy']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': '“It is not a lack of love, but a lack of friendship that makes unhappy marriages.”', 'autor': 'Friedrich Nietzsche', 'tags': ['friendship', 'lack-of-friendship', 'lack-of-love', 'love', 'marriage', 'unhappy-marriage']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': '“Good friends, good books, and a sleepy conscience: this is the ideal life.”', 'autor': 'Mark Twain', 'tags': ['books', 'contentment', 'friends', 'friendship', 'life']}
2026-03-04 21:53:10 [scrapy.core.scraper] DEBUG: Scraped from <200 https://quotes.toscrape.com/page/2/>
{'texto': '“Life is what happens to us while we are making other plans.”', 'autor': 'Allen Saunders', 'tags': ['fate', 'life', 'misattributed-john-lennon', 'planning', 'plans']}
2026-03-04 21:53:10 [scrapy.core.engine] INFO: Closing spider (finished)
2026-03-04 21:53:10 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 684,
 'downloader/request_count': 3,
 'downloader/request_method_count/GET': 3,
 'downloader/response_bytes': 25576,
 'downloader/response_count': 3,
 'downloader/response_status_count/200': 2,
 'downloader/response_status_count/404': 1,
 'elapsed_time_seconds': 2.88309,
 'finish_reason': 'finished',
 'finish_time': datetime.datetime(2026, 3, 5, 0, 53, 10, 957728, tzinfo=datetime.timezone.utc),
 'item_scraped_count': 20,
 'items_per_minute': 600.0,
 'log_count/DEBUG': 23,
 'log_count/INFO': 3,
 'response_received_count': 3,
 'responses_per_minute': 90.0,
 'robotstxt/request_count': 1,
 'robotstxt/response_count': 1,
 'robotstxt/response_status_count/404': 1,
 'scheduler/dequeued': 2,
 'scheduler/dequeued/memory': 2,
 'scheduler/enqueued': 2,
 'scheduler/enqueued/memory': 2,
 'start_time': datetime.datetime(2026, 3, 5, 0, 53, 8, 74638, tzinfo=datetime.timezone.utc)}
2026-03-04 21:53:10 [scrapy.core.engine] INFO: Spider closed (finished)